# Component Analysis H1 and H2 Detailed Results
Authoritative source for Section 4.1 (H1) and Section 4.2/4.5 (H2) of the thesis.
Run all cells top to bottom.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import warnings, os
warnings.filterwarnings('ignore')

BASE = r'D:\Bachelor Project Code/Code'

rav    = pd.read_csv(os.path.join(BASE, 'ravdess_features_norm.csv'))
adam   = pd.read_csv(os.path.join(BASE, 'adam_features_norm.csv'))
hope   = pd.read_csv(os.path.join(BASE, 'hope_features_norm.csv'))
ryan   = pd.read_csv(os.path.join(BASE, 'ryan_features_norm.csv'))
lucius = pd.read_csv(os.path.join(BASE, 'lucius_features_norm.csv'))
ivy    = pd.read_csv(os.path.join(BASE, 'ivy_features_norm.csv'))
sarah  = pd.read_csv(os.path.join(BASE, 'sarah_features_norm.csv'))
hume_r = pd.read_csv(os.path.join(BASE, 'hume_ravdess_results.csv'))
hume_e = pd.read_csv(os.path.join(BASE, 'hume_elevenlabs_results.csv'))
sb_r   = pd.read_csv(os.path.join(BASE, 'sb_ravdess_results.csv'))
sb_e   = pd.read_csv(os.path.join(BASE, 'sb_elevenlabs_results.csv'))

# Natural mode only
ai_nat = pd.concat(
    [v[v['mode'].str.lower().isin(['n', 'natural'])]
     for v in [adam, hope, ryan, lucius, ivy, sarah]],
    ignore_index=True
)

# SpeechBrain correct column fix
sb_r['correct_b'] = sb_r['correct'].map({True: True, False: False})
sb_e['correct_b'] = sb_e['correct'].map({True: True, False: False})

EMOTIONS = ['angry', 'disgust', 'fearful', 'happy', 'neutral', 'sad']

def cohens_d(a, b):
    psd = np.sqrt((a.std()**2 + b.std()**2) / 2)
    return (a.mean() - b.mean()) / psd if psd > 0 else 0.0

print(f'RAVDESS: {len(rav)} files')
print(f'AI Natural mode: {len(ai_nat)} files')
print(f'Hume RAVDESS: {len(hume_r)} | Hume ElevenLabs: {len(hume_e)}')
print(f'SpeechBrain RAVDESS: {len(sb_r)} | SpeechBrain ElevenLabs: {len(sb_e)}')

RAVDESS: 1440 files
AI Natural mode: 144 files
Hume RAVDESS: 1055 | Hume ElevenLabs: 240
SpeechBrain RAVDESS: 1056 | SpeechBrain ElevenLabs: 240


## H1 Overall feature differences (Section 4.1)

In [3]:
KEY_FEATURES = [
    ('speech_rate',            'Speech rate'),
    ('voiced_fraction',        'Voiced fraction'),
    ('pause_count',            'Pause count'),
    ('spectral_centroid_mean', 'Spectral centroid (Hz)'),
    ('pitch_range',            'Pitch range (Hz)'),
    ('hnr_mean',               'HNR (dB)'),
]

print(f"{'Feature':28}  {'Human mean':>12}  {'AI mean':>10}  {'Cohen d':>9}")
print('-' * 70)
for feat, label in KEY_FEATURES:
    h = rav[feat].dropna()
    a = ai_nat[feat].dropna()
    d = cohens_d(h, a)
    print(f"  {label:26}  {h.mean():12.4f}  {a.mean():10.4f}  d={d:+.3f}")

print()
h_sr = rav['speech_rate'].mean(); a_sr = ai_nat['speech_rate'].mean()
print(f"Speech rate: AI is {a_sr/h_sr:.2f}x faster  ({a_sr:.4f} vs {h_sr:.4f})")
h_pc = rav['pause_count'].mean(); a_pc = ai_nat['pause_count'].mean()
print(f"Pause count: human has {h_pc/a_pc:.1f}x more pauses  ({h_pc:.3f} vs {a_pc:.3f})")
h_sc = rav['spectral_centroid_mean'].mean(); a_sc = ai_nat['spectral_centroid_mean'].mean()
print(f"Spectral centroid: human={h_sc:.0f} Hz  AI={a_sc:.0f} Hz")
h_pr = rav['pitch_range'].mean(); a_pr = ai_nat['pitch_range'].mean()
print(f"Pitch range: human={h_pr:.0f} Hz  AI={a_pr:.0f} Hz")
h_hnr = rav['hnr_mean'].mean(); a_hnr = ai_nat['hnr_mean'].mean()
print(f"HNR: human={h_hnr:.2f} dB  AI={a_hnr:.2f} dB  (AI higher = cleaner, not noisier)")

Feature                         Human mean     AI mean    Cohen d
----------------------------------------------------------------------
  Speech rate                       0.4623      0.8132  d=-3.890
  Voiced fraction                   0.3716      0.5626  d=-1.399
  Pause count                       0.5208      0.1667  d=+0.522
  Spectral centroid (Hz)         5560.1793   3102.1778  d=+3.426
  Pitch range (Hz)                162.7249    104.6427  d=+0.746
  HNR (dB)                         10.0162     10.7016  d=-0.214

Speech rate: AI is 1.76x faster  (0.8132 vs 0.4623)
Pause count: human has 3.1x more pauses  (0.521 vs 0.167)
Spectral centroid: human=5560 Hz  AI=3102 Hz
Pitch range: human=163 Hz  AI=105 Hz
HNR: human=10.02 dB  AI=10.70 dB  (AI higher = cleaner, not noisier)


## H1 Cohen's d per feature (all 21 features)

In [4]:
ALL_FEATURES = [
    'pitch_mean','pitch_std','pitch_range','pitch_max','pitch_min',
    'voiced_fraction','speech_rate','pause_count','pause_rate','duration',
    'energy_mean','energy_std','energy_range',
    'spectral_centroid_mean','spectral_centroid_std','spectral_rolloff_mean',
    'zcr_mean','hfe_ratio','hnr_mean','jitter_local','shimmer_local',
]

results = []
for feat in ALL_FEATURES:
    if feat in rav.columns and feat in ai_nat.columns:
        h = rav[feat].dropna()
        a = ai_nat[feat].dropna()
        d = cohens_d(h, a)
        _, pval = ttest_ind(h, a)
        mag = 'large' if abs(d) > 0.8 else 'medium' if abs(d) > 0.5 else 'small'
        results.append({'feature': feat, 'd': d, 'abs_d': abs(d),
                        'p': pval, 'magnitude': mag})

df_d = pd.DataFrame(results).sort_values('abs_d', ascending=False)
print(f"{'Feature':28}  {'d':>8}  {'|d|':>6}  {'Magnitude':>10}")
print('-' * 60)
for _, row in df_d.iterrows():
    print(f"  {row['feature']:28}  {row['d']:+8.3f}  {row['abs_d']:6.3f}  {row['magnitude']:>10}")

Feature                              d     |d|   Magnitude
------------------------------------------------------------
  duration                        +5.019   5.019       large
  speech_rate                     -3.890   3.890       large
  spectral_rolloff_mean           +3.428   3.428       large
  spectral_centroid_mean          +3.426   3.426       large
  energy_mean                     -3.071   3.071       large
  energy_std                      +2.899   2.899       large
  spectral_centroid_std           +2.274   2.274       large
  energy_range                    +1.812   1.812       large
  voiced_fraction                 -1.399   1.399       large
  shimmer_local                   +0.838   0.838       large
  pitch_std                       +0.784   0.784      medium
  pitch_max                       +0.760   0.760      medium
  pitch_range                     +0.746   0.746      medium
  pitch_mean                      +0.674   0.674      medium
  pause_count             

## H1 Pause count per emotion (disgust extreme case, supports Section 4.5)

In [5]:
print("PAUSE COUNT per emotion — human vs AI Natural")
print(f"  {'Emotion':10}  {'Human':>10}  {'AI':>8}  {'Ratio (H/AI)':>14}")
print('-' * 50)
for e in EMOTIONS:
    h = rav[rav['emotion'] == e]['pause_count'].mean()
    a = ai_nat[ai_nat['emotion'] == e]['pause_count'].mean()
    ratio = f"{h/a:.1f}x" if a > 0 else "inf"
    print(f"  {e:10}  {h:10.3f}  {a:8.3f}  {ratio:>14}")

print()
print("Disgust: human produces ~12x more pauses than AI.")
print("Pauses in human disgust signal revulsion and reluctance.")
print("AI disgust produces continuous fluent speech — acoustically close to neutral.")

PAUSE COUNT per emotion — human vs AI Natural
  Emotion          Human        AI    Ratio (H/AI)
--------------------------------------------------
  angry            0.557     0.625            0.9x
  disgust          0.995     0.083           11.9x
  fearful          0.578     0.042           13.9x
  happy            0.276     0.042            6.6x
  neutral          0.177     0.042            4.3x
  sad              0.688     0.167            4.1x

Disgust: human produces ~12x more pauses than AI.
Pauses in human disgust signal revulsion and reluctance.
AI disgust produces continuous fluent speech — acoustically close to neutral.


## H1 HNR paradox: AI is cleaner, not noisier (Section 4.1)

In [6]:
print("HNR per emotion: AI has HIGHER HNR (more harmonic) than human for most emotions")
print(f"  {'Emotion':10}  {'Human HNR':>12}  {'AI HNR':>10}  {'Diff (AI-H)':>12}")
print('-' * 52)
for e in EMOTIONS:
    h = rav[rav['emotion'] == e]['hnr_mean'].mean()
    a = ai_nat[ai_nat['emotion'] == e]['hnr_mean'].mean()
    print(f"  {e:10}  {h:12.2f}  {a:10.2f}  {a-h:+12.2f}")

print()
print("AI failure on disgust is NOT due to noisy synthesis.")
print("AI is too clean (high HNR, no breathiness) to convey negative emotions.")

HNR per emotion: AI has HIGHER HNR (more harmonic) than human for most emotions
  Emotion        Human HNR      AI HNR   Diff (AI-H)
----------------------------------------------------
  angry               9.36       10.39         +1.03
  disgust             8.75       10.30         +1.55
  fearful            10.54       10.62         +0.08
  happy              10.74       12.18         +1.44
  neutral            10.76       10.11         -0.65
  sad                10.43       10.62         +0.18

AI failure on disgust is NOT due to noisy synthesis.
AI is too clean (high HNR, no breathiness) to convey negative emotions.


## H1 Anomalies: Lucius angry and Hope fearful exceed human pitch range

In [7]:
lucius_angry  = lucius[lucius['emotion'] == 'angry']['pitch_range'].mean()
hope_fearful  = hope[(hope['emotion'] == 'fearful') &
                     (hope['mode'].str.lower().isin(['n','natural']))]['pitch_range'].mean()
human_angry   = rav[rav['emotion'] == 'angry']['pitch_range'].mean()
human_fearful = rav[rav['emotion'] == 'fearful']['pitch_range'].mean()

print(f"Human angry pitch_range:  {human_angry:.1f} Hz")
print(f"Lucius angry pitch_range: {lucius_angry:.1f} Hz  ({lucius_angry/human_angry:.2f}x — EXCEEDS human)")
print()
print(f"Human fearful pitch_range: {human_fearful:.1f} Hz")
print(f"Hope fearful pitch_range:  {hope_fearful:.1f} Hz  ({hope_fearful/human_fearful:.2f}x — EXCEEDS human)")
print()
print("Both are genuine model behaviors retained in the dataset.")
print("They show ElevenLabs v3 can overshoot as well as undershoot human norms.")

Human angry pitch_range:  174.5 Hz
Lucius angry pitch_range: 231.9 Hz  (1.33x — EXCEEDS human)

Human fearful pitch_range: 162.2 Hz
Hope fearful pitch_range:  236.4 Hz  (1.46x — EXCEEDS human)

Both are genuine model behaviors retained in the dataset.
They show ElevenLabs v3 can overshoot as well as undershoot human norms.


## H2 Overall accuracy: Human vs AI voice (Section 4.2)

In [8]:
hume_h = hume_r['correct'].mean() * 100
hume_a = hume_e['correct'].mean() * 100
sb_r_eval = sb_r[sb_r['correct_b'].notna()]
sb_e_eval = sb_e[sb_e['correct_b'].notna()]
sb_h = sb_r_eval['correct_b'].mean() * 100
sb_a = sb_e_eval['correct_b'].mean() * 100

print("H2: OVERALL ACCURACY — HUMAN vs AI VOICE")
print(f"  Hume AI:      Human={hume_h:.1f}%  AI={hume_a:.1f}%  Gap={hume_h-hume_a:+.1f} pp")
print(f"  SpeechBrain:  Human={sb_h:.1f}%  AI={sb_a:.1f}%  Gap={sb_h-sb_a:+.1f} pp")
print()
print("H2 REJECTED: both detectors perform WORSE on AI voice.")
print()
print(f"SpeechBrain note: evaluated on {len(sb_r_eval)}/{len(sb_r)} RAVDESS files")
print("(fearful and disgust excluded — outside SpeechBrain's 4-class model)")

H2: OVERALL ACCURACY — HUMAN vs AI VOICE
  Hume AI:      Human=42.0%  AI=29.2%  Gap=+12.8 pp
  SpeechBrain:  Human=42.9%  AI=35.6%  Gap=+7.2 pp

H2 REJECTED: both detectors perform WORSE on AI voice.

SpeechBrain note: evaluated on 672/1056 RAVDESS files
(fearful and disgust excluded — outside SpeechBrain's 4-class model)


## H2 Per-emotion accuracy breakdown

In [9]:
print("HUME AI — PER-EMOTION ACCURACY")
print(f"  {'Emotion':10}  {'Human':>9}  {'AI':>9}  {'Gap':>9}")
for e in EMOTIONS:
    h = hume_r[hume_r['emotion'] == e]['correct'].mean() * 100
    a = hume_e[hume_e['emotion'] == e]['correct'].mean() * 100
    n_h = (hume_r['emotion'] == e).sum()
    n_a = (hume_e['emotion'] == e).sum()
    print(f"  {e:10}  {h:7.1f}% (n={n_h})  {a:7.1f}% (n={n_a})  {h-a:+7.1f} pp")

print()
print("SPEECHBRAIN — PER-EMOTION ACCURACY")
print(f"  {'Emotion':10}  {'Human':>9}  {'AI':>9}  {'Gap':>9}")
for e in EMOTIONS:
    h = sb_r[sb_r['emotion'] == e]['correct_b']
    a = sb_e[sb_e['emotion'] == e]['correct_b']
    h_acc = h.mean() * 100 if h.notna().sum() > 0 else float('nan')
    a_acc = a.mean() * 100 if a.notna().sum() > 0 else float('nan')
    gap = h_acc - a_acc if not (np.isnan(h_acc) or np.isnan(a_acc)) else float('nan')
    tag = "  [not evaluable - outside 4-class model]" if np.isnan(h_acc) else ""
    gap_str = f"{gap:+7.1f} pp" if not np.isnan(gap) else "      n/a"
    print(f"  {e:10}  {h_acc:7.1f}%         {a_acc:7.1f}%        {gap_str}{tag}")

HUME AI — PER-EMOTION ACCURACY
  Emotion         Human         AI        Gap
  angry          44.8% (n=192)     32.5% (n=40)    +12.3 pp
  disgust         0.0% (n=192)      0.0% (n=40)     +0.0 pp
  fearful        65.6% (n=192)     40.0% (n=40)    +25.6 pp
  happy          40.3% (n=191)     27.5% (n=40)    +12.8 pp
  neutral        96.9% (n=96)     57.5% (n=40)    +39.4 pp
  sad            31.8% (n=192)     17.5% (n=40)    +14.3 pp

SPEECHBRAIN — PER-EMOTION ACCURACY
  Emotion         Human         AI        Gap
  angry          95.3%            75.0%          +20.3 pp
  disgust         nan%             nan%              n/a  [not evaluable - outside 4-class model]
  fearful         nan%             nan%              n/a  [not evaluable - outside 4-class model]
  happy          17.2%             0.0%          +17.2 pp
  neutral        60.4%            65.0%           -4.6 pp
  sad             7.3%             2.5%           +4.8 pp


## H2 Hume neutral bias quantification (key finding for Section 4.5)

In [10]:
print("HUME: % PREDICTED AS NEUTRAL — AI vs HUMAN VOICE")
print(f"  {'Emotion':10}  {'AI → neutral':>14}  {'Human → neutral':>17}  {'Diff':>8}")
for e in EMOTIONS:
    ai_sub  = hume_e[hume_e['emotion'] == e]
    hum_sub = hume_r[hume_r['emotion'] == e]
    ai_n  = (ai_sub['hume_predicted'] == 'neutral').mean() * 100
    hum_n = (hum_sub['hume_predicted'] == 'neutral').mean() * 100
    print(f"  {e:10}  {ai_n:12.1f}%  {hum_n:15.1f}%  {ai_n-hum_n:+6.1f} pp")

print()
print("KEY: SAD (most affected emotion)")
ai_sad  = hume_e[hume_e['emotion'] == 'sad']
hum_sad = hume_r[hume_r['emotion'] == 'sad']
ai_to_n  = (ai_sad['hume_predicted'] == 'neutral').sum()
hum_to_n = (hum_sad['hume_predicted'] == 'neutral').sum()
print(f"  AI sad predicted as neutral:    {ai_to_n}/{len(ai_sad)} = {ai_to_n/len(ai_sad)*100:.1f}%")
print(f"  Human sad predicted as neutral: {hum_to_n}/{len(hum_sad)} = {hum_to_n/len(hum_sad)*100:.1f}%")
print()
print("The neutral bias is specific to AI voice, not a general Hume limitation.")
print("For sadness specifically, AI triggers neutral 80% of the time vs 46.9% for human.")

HUME: % PREDICTED AS NEUTRAL — AI vs HUMAN VOICE
  Emotion       AI → neutral    Human → neutral      Diff
  angry               12.5%              9.4%    +3.1 pp
  disgust             37.5%             16.7%   +20.8 pp
  fearful             27.5%             10.9%   +16.6 pp
  happy               35.0%             19.9%   +15.1 pp
  neutral             57.5%             96.9%   -39.4 pp
  sad                 80.0%             46.9%   +33.1 pp

KEY: SAD (most affected emotion)
  AI sad predicted as neutral:    32/40 = 80.0%
  Human sad predicted as neutral: 90/192 = 46.9%

The neutral bias is specific to AI voice, not a general Hume limitation.
For sadness specifically, AI triggers neutral 80% of the time vs 46.9% for human.


## H2 Hume disgust investigation: score_disgust analysis

In [11]:
print("HUME DISGUST: does Hume assign disgust scores at all?")
print()
ai_d  = hume_e[hume_e['emotion'] == 'disgust']
hum_d = hume_r[hume_r['emotion'] == 'disgust']

print("AI disgust files:")
print(f"  score_disgust: mean={ai_d['score_disgust'].mean():.4f}  max={ai_d['score_disgust'].max():.4f}")
print(f"  score_neutral: mean={ai_d['score_neutral'].mean():.4f}")
print(f"  score_sad:     mean={ai_d['score_sad'].mean():.4f}")
print(f"  hume_predicted distribution: {ai_d['hume_predicted'].value_counts().to_dict()}")

print()
print("Human disgust files:")
print(f"  score_disgust: mean={hum_d['score_disgust'].mean():.4f}  max={hum_d['score_disgust'].max():.4f}")
print(f"  hume_predicted distribution: {hum_d['hume_predicted'].value_counts().to_dict()}")

print()
print("CONCLUSION: Hume assigns non-zero score_disgust but it never wins.")
print("Neutral and sad consistently score higher → hume_predicted is never disgust.")
print("This is a detector limitation, not an absence of disgust cues.")
print("Human listeners correctly identify human disgust at 46.8% (see survey_analysis.ipynb).")

HUME DISGUST: does Hume assign disgust scores at all?

AI disgust files:
  score_disgust: mean=0.1103  max=0.3567
  score_neutral: mean=0.3892
  score_sad:     mean=0.3750
  hume_predicted distribution: {'neutral': 15, 'sad': 14, 'angry': 6, 'happy': 4, 'fearful': 1}

Human disgust files:
  score_disgust: mean=0.1095  max=0.4723
  hume_predicted distribution: {'sad': 73, 'angry': 35, 'neutral': 32, 'fearful': 24, 'happy': 22, 'error': 6}

CONCLUSION: Hume assigns non-zero score_disgust but it never wins.
Neutral and sad consistently score higher → hume_predicted is never disgust.
This is a detector limitation, not an absence of disgust cues.
Human listeners correctly identify human disgust at 46.8% (see survey_analysis.ipynb).


## H2 SpeechBrain exception: neutral AI outperforms neutral human

In [12]:
sb_n_h = sb_r[sb_r['emotion'] == 'neutral']['correct_b']
sb_n_a = sb_e[sb_e['emotion'] == 'neutral']['correct_b']

print("SPEECHBRAIN: NEUTRAL — the only emotion where AI beats human")
print(f"  Human neutral: {sb_n_h.mean()*100:.1f}%  ({int(sb_n_h.sum())}/{sb_n_h.notna().sum()})")
print(f"  AI neutral:    {sb_n_a.mean()*100:.1f}%  ({int(sb_n_a.sum())}/{sb_n_a.notna().sum()})")
print()
print("AI neutral is MORE recognizable than human neutral for SpeechBrain.")
print("Consistent with H1: AI neutral is acoustically over-clean and prototypical.")

SPEECHBRAIN: NEUTRAL — the only emotion where AI beats human
  Human neutral: 60.4%  (58/96)
  AI neutral:    65.0%  (26/40)

AI neutral is MORE recognizable than human neutral for SpeechBrain.
Consistent with H1: AI neutral is acoustically over-clean and prototypical.
